# YgoMaster-PackBuilder — Project Cross-Validation Tests

Notebook di test manuale/esplorativo per la **cross-validation**
tra:

- `config/project.yaml`
- `config/policies.yaml`

Obiettivi:

- verificare che la default opening policy esista;
- verificare gli override dei singoli pack;
- verificare l'ereditarietà tramite `None`;
- controllare che gli errori cross-file vengano raccolti correttamente.

## 1. Ambiente e root della repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository. "
        "Esegui il notebook dalla root del progetto oppure dalla cartella tests/."
    )

manifest_path = repo_root / "config" / "project.yaml"
policies_path = repo_root / "config" / "policies.yaml"

assert manifest_path.exists(), f"Manifest non trovato: {manifest_path}"
assert policies_path.exists(), f"Policies config non trovato: {policies_path}"

print("Repository root:", repo_root)
print("Manifest path:", manifest_path)
print("Policies path:", policies_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Manifest path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\project.yaml
Policies path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\policies.yaml


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.manifest import load_project_manifest
from ygomaster_packbuilder.policies import load_policies_config

from ygomaster_packbuilder.validation import (
    ValidationReport,
    validate_project_configuration,
)

print("Cross-validation imports OK")

Cross-validation imports OK


## 3. Helper per visualizzare il report

In [3]:
def show_report(report: ValidationReport) -> None:
    print("Valid:", report.is_valid)
    print("Errors:", len(report.errors))
    print("Warnings:", len(report.warnings))
    print()

    for issue in report.issues:
        print(
            f"[{issue.severity.value.upper()}] "
            f"{issue.code} @ {issue.path}"
        )
        print(" ", issue.message)
        print()


def issue_codes(report: ValidationReport) -> list[str]:
    return [issue.code for issue in report.issues]


def assert_has_code(report: ValidationReport, code: str) -> None:
    assert code in issue_codes(report), (
        f"Expected validation code {code!r}, "
        f"found {issue_codes(report)!r}"
    )

print("Helpers ready")

Helpers ready


## 4. Caricamento configurazioni reali

In [4]:
manifest = load_project_manifest(manifest_path)
policies = load_policies_config(policies_path)

print("Project:", manifest.settings.name)
print("Policies:", [policy.key for policy in policies.policies])

Project: Master Duel Custom League
Policies: ['configuration-a']


## 5. Cross-validation reale

In [5]:
report = validate_project_configuration(
    manifest,
    policies,
)

show_report(report)

assert isinstance(report, ValidationReport)
assert report.is_valid
assert report.errors == []

print("Current project configuration is valid")

Valid: True
Errors: 0
Warnings: 0

Current project configuration is valid


## 6. Default opening policy esistente

La default dichiarata nel manifest deve essere presente in `PoliciesConfig`.


In [6]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert report.is_valid
print("Existing default policy accepted")

Valid: True
Errors: 0
Warnings: 0

Existing default policy accepted


## 7. Default opening policy inesistente

Regola attesa:

```text
unknown-default-opening-policy
```


In [7]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

case_manifest.settings.default_opening_policy = "missing-policy"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unknown-default-opening-policy")

Valid: False
Errors: 1
Warnings: 0

[ERROR] unknown-default-opening-policy @ settings.default_opening_policy
  Default opening policy 'missing-policy' is not defined.



## 8. Override assente (`None`)

Un pack senza override deve ereditare la default e non produrre errori.


In [8]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

case_manifest.players[0].main_pack.opening_policy = None

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert report.is_valid
print("None override correctly inherits the default policy")

Valid: True
Errors: 0
Warnings: 0

None override correctly inherits the default policy


## 9. Override valido

Un override esplicito che punta a una policy esistente deve essere accettato.


In [9]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

existing_policy = case_policies.policies[0].key
case_manifest.players[0].main_pack.opening_policy = existing_policy

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert report.is_valid
print("Valid explicit override accepted")

Valid: True
Errors: 0
Warnings: 0

Valid explicit override accepted


## 10. Override inesistente

Regola attesa:

```text
unknown-opening-policy
```


In [10]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

case_manifest.players[0].main_pack.opening_policy = "missing-policy"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unknown-opening-policy")

Valid: False
Errors: 1
Warnings: 0

[ERROR] unknown-opening-policy @ players[0].main_pack.opening_policy
  Opening policy 'missing-policy' is not defined.



## 11. Override inesistente su mini-pack

Verifichiamo che il traversal includa anche i mini-pack.


In [11]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

assert case_manifest.players[0].mini_packs

case_manifest.players[0].mini_packs[0].opening_policy = "missing-mini-policy"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unknown-opening-policy")

matching = [
    issue
    for issue in report.issues
    if issue.code == "unknown-opening-policy"
]

assert matching
assert ".mini_packs[0].opening_policy" in matching[0].path

Valid: False
Errors: 1
Warnings: 0

[ERROR] unknown-opening-policy @ players[0].mini_packs[0].opening_policy
  Opening policy 'missing-mini-policy' is not defined.



## 12. Override inesistente su shared pack

Verifichiamo che il traversal includa anche i shared pack.


In [12]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

assert case_manifest.shared_packs

case_manifest.shared_packs[0].opening_policy = "missing-shared-policy"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unknown-opening-policy")

matching = [
    issue
    for issue in report.issues
    if issue.code == "unknown-opening-policy"
]

assert matching
assert matching[0].path.startswith("shared_packs[")

Valid: False
Errors: 1
Warnings: 0

[ERROR] unknown-opening-policy @ shared_packs[0].opening_policy
  Opening policy 'missing-shared-policy' is not defined.



## 13. Policy rimossa dal file delle policies

Questo simula una situazione in cui il manifest continua a referenziare una policy
che non esiste più in `policies.yaml`.


In [13]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

default_key = case_manifest.settings.default_opening_policy

case_policies.policies = [
    policy
    for policy in case_policies.policies
    if policy.key != default_key
]

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unknown-default-opening-policy")

Valid: False
Errors: 1
Warnings: 0

[ERROR] unknown-default-opening-policy @ settings.default_opening_policy
  Default opening policy 'configuration-a' is not defined.



## 14. Più errori contemporaneamente

La cross-validation deve raccogliere sia l'errore sulla default sia gli errori sugli override.


In [14]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

case_manifest.settings.default_opening_policy = "missing-default"
case_manifest.players[0].main_pack.opening_policy = "missing-main"

if case_manifest.players[0].mini_packs:
    case_manifest.players[0].mini_packs[0].opening_policy = "missing-mini"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

show_report(report)

codes = issue_codes(report)

assert "unknown-default-opening-policy" in codes
assert codes.count("unknown-opening-policy") >= 2

print("Multiple cross-validation errors collected")

Valid: False
Errors: 3
Warnings: 0

[ERROR] unknown-default-opening-policy @ settings.default_opening_policy
  Default opening policy 'missing-default' is not defined.

[ERROR] unknown-opening-policy @ players[0].main_pack.opening_policy
  Opening policy 'missing-main' is not defined.

[ERROR] unknown-opening-policy @ players[0].mini_packs[0].opening_policy
  Opening policy 'missing-mini' is not defined.

Multiple cross-validation errors collected


## 15. Controllo dei path

I path devono indicare il punto logico del manifest che contiene il riferimento errato.


In [15]:
case_manifest = deepcopy(manifest)
case_policies = deepcopy(policies)

case_manifest.players[0].main_pack.opening_policy = "not-defined"

report = validate_project_configuration(
    case_manifest,
    case_policies,
)

matching = [
    issue
    for issue in report.issues
    if issue.code == "unknown-opening-policy"
]

assert matching

issue = matching[0]

print("Code:", issue.code)
print("Path:", issue.path)
print("Message:", issue.message)

assert issue.path.endswith(".opening_policy")

Code: unknown-opening-policy
Path: players[0].main_pack.opening_policy
Message: Opening policy 'not-defined' is not defined.
